# Phase 3 - Query Decomposition RAG

This notebook continues from the completed Standard RAG Dev20 baseline and builds **Query Decomposition RAG step by step**.

Pipeline:

**Original Question -> DeepSeek Decomposition -> Sub-questions -> BM25 + DPR/FAISS for each Sub-question -> Merge & Deduplicate -> BGE Reranker -> Top-10 Evidence -> DeepSeek Final Answer -> EM/F1 + Development Retrieval Metrics**

It reuses the same BeerQA Dev20 corpus and saved retrieval artifacts so the comparison with Standard RAG remains consistent.

## Step 1 - Environment and Paths

If needed, install once:

`pip install rank-bm25 transformers torch faiss-cpu sentencepiece openai`

In [51]:
from pathlib import Path
from collections import Counter
import os
import json
import re
import string
import getpass
import numpy as np
import torch
import faiss

from rank_bm25 import BM25Okapi
from transformers import (
    DPRQuestionEncoder,
    DPRQuestionEncoderTokenizer,
    AutoTokenizer,
    AutoModelForSequenceClassification,
)
from openai import OpenAI

BEERQA_DEV_PATH = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\beerqa_dev_v1.0.json"
)

OUTPUT_DIR = Path(
    r"C:\Users\Wei Khang\Documents\Master\FYP\processed_data"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEERQA_CORPUS_PATH = OUTPUT_DIR / "beerqa_dev20_wikipedia_corpus.jsonl"
BEERQA_DPR_EMBEDDINGS_PATH = OUTPUT_DIR / "beerqa_dev20_dpr_embeddings.npy"
BEERQA_FAISS_PATH = OUTPUT_DIR / "beerqa_dev20.faiss"

QD_DEV5_PATH = OUTPUT_DIR / "query_decomposition_rag_dev5_results.jsonl"
QD_DEV20_PATH = OUTPUT_DIR / "query_decomposition_rag_dev20_results.jsonl"
QD_DEV20_EVAL_PATH = OUTPUT_DIR / "query_decomposition_rag_dev20_evaluated.jsonl"
STANDARD_EVAL_PATH = OUTPUT_DIR / "standard_rag_dev20_evaluated.jsonl"

print("BeerQA dev exists:", BEERQA_DEV_PATH.exists())
print("BeerQA corpus exists:", BEERQA_CORPUS_PATH.exists())
print("DPR embeddings exist:", BEERQA_DPR_EMBEDDINGS_PATH.exists())
print("FAISS index exists:", BEERQA_FAISS_PATH.exists())

BeerQA dev exists: True
BeerQA corpus exists: True
DPR embeddings exist: True
FAISS index exists: True


## Step 2 - Shared Helpers

In [52]:
def normalize_title(title):
    return " ".join(
        str(title).replace("_", " ").strip().lower().split()
    )


def bm25_tokenize(text):
    return re.findall(r"\b\w+\b", str(text).lower())

## Step 3 - Load BeerQA Dev20 and the Saved Development Corpus

In [53]:
with open(BEERQA_DEV_PATH, "r", encoding="utf-8") as f:
    beerqa_dev = json.load(f)

beerqa_records = beerqa_dev["data"]
beerqa_sample = beerqa_records[:20]

print("Total BeerQA dev records:", len(beerqa_records))
print("Development sample:", len(beerqa_sample))

Total BeerQA dev records: 14121
Development sample: 20


In [54]:
if not BEERQA_CORPUS_PATH.exists():
    raise FileNotFoundError(
        "beerqa_dev20_wikipedia_corpus.jsonl is missing. "
        "Run the Standard RAG preparation notebook first."
    )

beerqa_corpus = []
with open(BEERQA_CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        beerqa_corpus.append(json.loads(line))

passage_ids = [p["passage_id"] for p in beerqa_corpus]
word_counts = [len(p["text"].split()) for p in beerqa_corpus]

print("Loaded BeerQA passages:", len(beerqa_corpus))
print("Unique passage IDs:", len(set(passage_ids)))
print("Maximum words:", max(word_counts))

assert len(passage_ids) == len(set(passage_ids))
assert max(word_counts) <= 100
print("PASS: BeerQA corpus is ready.")

Loaded BeerQA passages: 1674
Unique passage IDs: 1674
Maximum words: 100
PASS: BeerQA corpus is ready.


## Step 4 - BM25 Retrieval

In [55]:
beerqa_bm25_documents = [
    chunk["title"] + " " + chunk["text"]
    for chunk in beerqa_corpus
]

beerqa_tokenized_corpus = [
    bm25_tokenize(document)
    for document in beerqa_bm25_documents
]

beerqa_bm25 = BM25Okapi(beerqa_tokenized_corpus)
print("BM25 passages indexed:", len(beerqa_tokenized_corpus))

BM25 passages indexed: 1674


In [56]:
def beerqa_bm25_search(query, top_k=10):
    scores = beerqa_bm25.get_scores(bm25_tokenize(query))
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = beerqa_corpus[idx]
        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "bm25_score": float(scores[idx]),
        })

    return results

## Step 5 - DPR Question Encoder and Saved FAISS Index

The passage embeddings are already saved, so this notebook only loads them and the FAISS index. It does not regenerate passage embeddings.

In [57]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

QUESTION_MODEL_NAME = "facebook/dpr-question_encoder-multiset-base"

question_tokenizer = DPRQuestionEncoderTokenizer.from_pretrained(
    QUESTION_MODEL_NAME
)
question_encoder = DPRQuestionEncoder.from_pretrained(
    QUESTION_MODEL_NAME
).to(device)
question_encoder.eval()

print("DPR question encoder loaded.")

Device: cpu


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 33305.57it/s]
[transformers] DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-multiset-base
Key                                             | Status     |  | 
------------------------------------------------+------------+--+-
question_encoder.bert_model.pooler.dense.bias   | UNEXPECTED |  | 
question_encoder.bert_model.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DPR question encoder loaded.


In [58]:
if not BEERQA_DPR_EMBEDDINGS_PATH.exists():
    raise FileNotFoundError(
        "beerqa_dev20_dpr_embeddings.npy is missing. "
        "Run the Standard RAG preparation notebook first."
    )

beerqa_passage_embeddings = np.load(BEERQA_DPR_EMBEDDINGS_PATH)

assert beerqa_passage_embeddings.shape[0] == len(beerqa_corpus)
assert beerqa_passage_embeddings.shape[1] == 768

print("Loaded embeddings:", beerqa_passage_embeddings.shape)

Loaded embeddings: (1674, 768)


In [59]:
if BEERQA_FAISS_PATH.exists():
    beerqa_faiss_index = faiss.read_index(str(BEERQA_FAISS_PATH))
else:
    beerqa_faiss_index = faiss.IndexFlatIP(
        beerqa_passage_embeddings.shape[1]
    )
    beerqa_faiss_index.add(
        beerqa_passage_embeddings.astype("float32")
    )
    faiss.write_index(beerqa_faiss_index, str(BEERQA_FAISS_PATH))

assert beerqa_faiss_index.ntotal == len(beerqa_corpus)
print("FAISS vectors:", beerqa_faiss_index.ntotal)

FAISS vectors: 1674


In [60]:
def beerqa_dpr_search(query, top_k=10):
    inputs = question_tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        max_length=256,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        query_embedding = (
            question_encoder(**inputs)
            .pooler_output
            .cpu()
            .numpy()
            .astype("float32")
        )

    scores, indices = beerqa_faiss_index.search(query_embedding, top_k)
    results = []

    for rank, (idx, score) in enumerate(
        zip(indices[0], scores[0]), start=1
    ):
        chunk = beerqa_corpus[idx]
        results.append({
            "rank": rank,
            "passage_id": chunk["passage_id"],
            "article_id": chunk["article_id"],
            "title": chunk["title"],
            "text": chunk["text"],
            "dpr_score": float(score),
        })

    return results

## Step 6 - Load `bge-reranker-large`

Query Decomposition can create a larger candidate pool from several sub-questions. The reranker scores the merged passages against the **original question** and keeps the final Top-10.

In [61]:
RERANKER_MODEL_NAME = "BAAI/bge-reranker-large"

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME
).to(device)
reranker_model.eval()

print("Reranker loaded:", RERANKER_MODEL_NAME)

Loading weights: 100%|██████████| 393/393 [00:01<00:00, 278.86it/s]


Reranker loaded: BAAI/bge-reranker-large


In [62]:
def rerank_candidates(query, candidates, top_k=10, batch_size=4):
    if not candidates:
        return []

    passages = [
        f"{candidate['title']}. {candidate['text']}"
        for candidate in candidates
    ]
    scores_all = []

    for start in range(0, len(passages), batch_size):
        batch_passages = passages[start:start + batch_size]
        batch_queries = [query] * len(batch_passages)

        inputs = reranker_tokenizer(
            batch_queries,
            batch_passages,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.inference_mode():
            outputs = reranker_model(**inputs)
            scores = outputs.logits.view(-1).float().cpu().tolist()

        scores_all.extend(scores)

    reranked = []
    for candidate, score in zip(candidates, scores_all):
        item = candidate.copy()
        item["reranker_score"] = float(score)
        reranked.append(item)

    reranked.sort(key=lambda x: x["reranker_score"], reverse=True)
    return reranked[:top_k]

## Step 7 - Connect DeepSeek

The API key is not stored in the notebook. Paste it when prompted.

In [63]:
if not os.getenv("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass(
        "Enter your DeepSeek API key: "
    )

client = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
)

DEEPSEEK_MODEL = "deepseek-v4-flash"
print("DeepSeek client ready.")

DeepSeek client ready.


# Query Decomposition RAG

## Step 8 - QD1: Decompose the Original Question

The decomposer returns 1-4 independently searchable sub-questions as JSON. If JSON parsing fails, the original question is used as a fallback and the fallback is recorded.

In [ ]:
def _parse_subquestions(raw_text, original_question, max_subquestions=4):
    cleaned = raw_text.strip() # remove whitespace
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE) # remove opening Markdown code fence (generate by DEEPSEEK)
    cleaned = re.sub(r"\s*```$", "", cleaned) # remove closing Markdown code fence (generate by DEEPSEEK)

    try:
        parsed = json.loads(cleaned) # store the json into dict
        if isinstance(parsed, dict):
            items = parsed.get("subquestions", [])
        elif isinstance(parsed, list):
            items = parsed
        else:
            items = []
    except json.JSONDecodeError:
        items = []

    subquestions = []
    for item in items:
        if not isinstance(item, str):
            continue
        q = " ".join(item.strip().split()) # remove whitespace beginning and end then separate the sentence into word using whitespace
        if q and q not in subquestions:
            subquestions.append(q)
        if len(subquestions) >= max_subquestions:
            break

    used_fallback = False
    if not subquestions:
        subquestions = [original_question]
        used_fallback = True

    return subquestions, used_fallback

In [65]:
def decompose_question(question, model=DEEPSEEK_MODEL, max_subquestions=4):
    system_prompt = """
You are a question decomposition component for an open-domain
multi-hop question answering system.

Break the original question into the minimum number of simpler,
independently searchable sub-questions needed to answer it.

Rules:
1. Return between 1 and 4 sub-questions.
2. Each sub-question must be answerable using a Wikipedia passage.
3. Preserve important entity names from the original question.
4. Do not answer the original question.
5. Do not provide explanations or reasoning.
6. If the original question is already atomic, return one equivalent sub-question.
7. Return ONLY valid JSON in exactly this format:

{"subquestions": ["...", "..."]}
""".strip()

    user_prompt = f"""
Original Question:
{question}

Return the decomposition JSON only.
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=250,
        temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},
    )

    raw_output = response.choices[0].message.content.strip()
    subquestions, used_fallback = _parse_subquestions(
        raw_output,
        original_question=question,
        max_subquestions=max_subquestions,
    )

    return {
        "subquestions": subquestions,
        "raw_output": raw_output,
        "used_fallback": used_fallback,
    }

### Step 8A - Test the Decomposition on One Real BeerQA Question

In [66]:
test_record = beerqa_sample[0]
test_question = test_record["question"]

print("Original Question:")
print(test_question)
print("\nGold Answer:")
print(test_record["answers"])

decomposition_test = decompose_question(test_question)

print("\nGenerated Sub-questions:")
for i, subq in enumerate(decomposition_test["subquestions"], start=1):
    print(f"{i}. {subq}")

print("\nFallback used:", decomposition_test["used_fallback"])

Original Question:
Which genus contains more species, Ortegocactus or Eschscholzia?

Gold Answer:
['Eschscholzia']

Generated Sub-questions:
1. How many species are in the genus Ortegocactus?
2. How many species are in the genus Eschscholzia?

Fallback used: False


## Step 9 - QD2: Hybrid Retrieval for One Sub-question

For each sub-question, retrieve BM25 Top-10 and DPR Top-10, merge the candidates, and remove duplicate passage IDs. Raw BM25 and DPR scores are not added together.

In [67]:
def hybrid_candidates_for_query(query, top_k_each=10):
    bm25_results = beerqa_bm25_search(query, top_k=top_k_each)
    dpr_results = beerqa_dpr_search(query, top_k=top_k_each)
    combined = {}

    for result in bm25_results:
        pid = result["passage_id"]
        combined[pid] = {
            "passage_id": pid,
            "article_id": result["article_id"],
            "title": result["title"],
            "text": result["text"],
            "bm25_rank": result["rank"],
            "dpr_rank": None,
            "bm25_score": result["bm25_score"],
            "dpr_score": None,
            "source": ["BM25"],
        }

    for result in dpr_results:
        pid = result["passage_id"]
        if pid not in combined:
            combined[pid] = {
                "passage_id": pid,
                "article_id": result["article_id"],
                "title": result["title"],
                "text": result["text"],
                "bm25_rank": None,
                "dpr_rank": result["rank"],
                "bm25_score": None,
                "dpr_score": result["dpr_score"],
                "source": ["DPR"],
            }
        else:
            combined[pid]["dpr_rank"] = result["rank"]
            combined[pid]["dpr_score"] = result["dpr_score"]
            combined[pid]["source"].append("DPR")

    return list(combined.values())

In [68]:
first_subquestion = decomposition_test["subquestions"][0]
subq_candidates = hybrid_candidates_for_query(first_subquestion, top_k_each=10)

print("Sub-question:", first_subquestion)
print("Unique hybrid candidates:", len(subq_candidates))

for i, item in enumerate(subq_candidates[:10], start=1):
    print(i, "|", item["passage_id"], "|", item["title"], "|", item["source"])

Sub-question: How many species are in the genus Ortegocactus?
Unique hybrid candidates: 18
1 | 2678316_0 | Ortegocactus | ['BM25', 'DPR']
2 | 250512_0 | Eschscholzia | ['BM25', 'DPR']
3 | 25614_23 | Race (human categorization) | ['BM25']
4 | 25614_101 | Race (human categorization) | ['BM25']
5 | 27105_19 | Leonard McCoy | ['BM25']
6 | 250512_1 | Eschscholzia | ['BM25']
7 | 560876_79 | Pharmaceutical industry | ['BM25']
8 | 560876_27 | Pharmaceutical industry | ['BM25']
9 | 25614_65 | Race (human categorization) | ['BM25']
10 | 560876_69 | Pharmaceutical industry | ['BM25']


## Step 10 - QD3: Retrieve for All Sub-questions and Merge Evidence

Candidates from every sub-question are combined into one shared pool. Duplicate passages are stored once, while retrieval provenance is kept in `retrieval_queries`.

In [69]:
def retrieve_decomposed_candidates(subquestions, top_k_each=10):
    merged = {}

    for subq_index, subquestion in enumerate(subquestions, start=1):
        candidates = hybrid_candidates_for_query(
            subquestion,
            top_k_each=top_k_each,
        )

        for candidate in candidates:
            pid = candidate["passage_id"]

            if pid not in merged:
                merged[pid] = {
                    "passage_id": candidate["passage_id"],
                    "article_id": candidate["article_id"],
                    "title": candidate["title"],
                    "text": candidate["text"],
                    "retrieval_queries": [],
                }

            merged[pid]["retrieval_queries"].append({
                "subquestion_index": subq_index,
                "subquestion": subquestion,
                "source": candidate["source"],
                "bm25_rank": candidate["bm25_rank"],
                "dpr_rank": candidate["dpr_rank"],
                "bm25_score": candidate["bm25_score"],
                "dpr_score": candidate["dpr_score"],
            })

    return list(merged.values())

In [70]:
merged_candidates_test = retrieve_decomposed_candidates(
    decomposition_test["subquestions"],
    top_k_each=10,
)

merged_ids = [item["passage_id"] for item in merged_candidates_test]

print("Number of sub-questions:", len(decomposition_test["subquestions"]))
print("Merged unique candidates:", len(merged_candidates_test))
print("Unique passage IDs:", len(set(merged_ids)))

assert len(merged_ids) == len(set(merged_ids))
print("PASS: No duplicate passages.")

Number of sub-questions: 2
Merged unique candidates: 22
Unique passage IDs: 22
PASS: No duplicate passages.


## Step 11 - QD4: Rerank the Merged Candidate Pool

All merged candidates are scored against the **original question** using the same BGE reranker. Keep the final Top-10 evidence.

In [71]:
reranked_test = rerank_candidates(
    query=test_question,
    candidates=merged_candidates_test,
    top_k=10,
    batch_size=4,
)

print("Final reranked passages:", len(reranked_test))

for rank, item in enumerate(reranked_test, start=1):
    print(
        rank,
        "|",
        item["passage_id"],
        "|",
        item["title"],
        "| reranker:",
        round(item["reranker_score"], 4),
    )

Final reranked passages: 10
1 | 250512_0 | Eschscholzia | reranker: 9.5229
2 | 2678316_0 | Ortegocactus | reranker: 2.71
3 | 250512_1 | Eschscholzia | reranker: 1.2289
4 | 250512_2 | Eschscholzia | reranker: 0.3138
5 | 12551_41 | Gymnastics | reranker: -7.4393
6 | 27105_19 | Leonard McCoy | reranker: -7.4977
7 | 12551_42 | Gymnastics | reranker: -7.7083
8 | 12551_53 | Gymnastics | reranker: -7.8278
9 | 12551_9 | Gymnastics | reranker: -7.8631
10 | 25614_65 | Race (human categorization) | reranker: -7.8711


## Step 12 - QD5: Build the Final Evidence Context

In [72]:
def build_qd_context(reranked_results, max_passages=10):
    blocks = []

    for i, result in enumerate(reranked_results[:max_passages], start=1):
        blocks.append(
            f"[Passage {i}]\n"
            f"Title: {result['title']}\n"
            f"Text: {result['text']}"
        )

    return "\n\n".join(blocks)

## Step 13 - QD6: Generate the Final Answer

In [73]:
def generate_qd_answer(question, subquestions, context, model=DEEPSEEK_MODEL):
    system_prompt = """
You are an open-domain question answering system.

Answer the original question using only the retrieved Wikipedia evidence.
The original question was decomposed into sub-questions to improve retrieval.

Rules:
1. Use the retrieved evidence for the final answer.
2. Use the sub-questions only to understand the information needs.
3. Do not invent unsupported information.
4. Return only the short factual final answer.
5. Do not explain the decomposition.
6. If the evidence is insufficient, return: INSUFFICIENT_EVIDENCE
""".strip()

    subquestion_text = "\n".join(
        f"{i}. {subq}"
        for i, subq in enumerate(subquestions, start=1)
    )

    user_prompt = f"""
Original Question:
{question}

Sub-questions:
{subquestion_text}

Retrieved Wikipedia Evidence:
{context}

Final Answer:
""".strip()

    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        max_tokens=100,
        temperature=0.0,
        extra_body={"thinking": {"type": "disabled"}},
    )

    return response.choices[0].message.content.strip()

## Step 14 - QD7: Complete `query_decomposition_rag()` Function

In [74]:
def query_decomposition_rag(
    question,
    top_k_each=10,
    final_top_k=10,
    reranker_batch_size=4,
    model=DEEPSEEK_MODEL,
):
    decomposition = decompose_question(question, model=model)
    subquestions = decomposition["subquestions"]

    merged_candidates = retrieve_decomposed_candidates(
        subquestions,
        top_k_each=top_k_each,
    )

    reranked_results = rerank_candidates(
        query=question,
        candidates=merged_candidates,
        top_k=final_top_k,
        batch_size=reranker_batch_size,
    )

    context = build_qd_context(
        reranked_results,
        max_passages=final_top_k,
    )

    generated_answer = generate_qd_answer(
        question=question,
        subquestions=subquestions,
        context=context,
        model=model,
    )

    return {
        "question": question,
        "subquestions": subquestions,
        "raw_decomposition": decomposition["raw_output"],
        "decomposition_fallback": decomposition["used_fallback"],
        "candidate_count_before_rerank": len(merged_candidates),
        "generated_answer": generated_answer,
        "retrieved_passages": reranked_results,
        "context": context,
    }

## Step 15 - QD8: Run One Real BeerQA Question End to End

This makes one DeepSeek call for decomposition and one DeepSeek call for final answer generation.

In [75]:
test_record = beerqa_sample[0]
qd_result = query_decomposition_rag(test_record["question"])

print("QUESTION:")
print(test_record["question"])
print("\nGOLD ANSWER:")
print(test_record["answers"])
print("\nSUB-QUESTIONS:")
for i, subq in enumerate(qd_result["subquestions"], start=1):
    print(f"{i}. {subq}")
print("\nCANDIDATES BEFORE RERANK:", qd_result["candidate_count_before_rerank"])
print("\nQD RAG ANSWER:")
print(qd_result["generated_answer"])
print("\nDECOMPOSITION FALLBACK:", qd_result["decomposition_fallback"])

QUESTION:
Which genus contains more species, Ortegocactus or Eschscholzia?

GOLD ANSWER:
['Eschscholzia']

SUB-QUESTIONS:
1. How many species are in the genus Ortegocactus?
2. How many species are in the genus Eschscholzia?

CANDIDATES BEFORE RERANK: 22

QD RAG ANSWER:
Eschscholzia

DECOMPOSITION FALLBACK: False


## Step 16 - EM and F1 Evaluation

In [76]:
def normalize_answer(text):
    text = str(text).lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def exact_match_score(prediction, ground_truth):
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score(prediction, ground_truth):
    prediction_tokens = normalize_answer(prediction).split()
    ground_truth_tokens = normalize_answer(ground_truth).split()

    if not prediction_tokens or not ground_truth_tokens:
        return float(prediction_tokens == ground_truth_tokens)

    common = Counter(prediction_tokens) & Counter(ground_truth_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(prediction_tokens)
    recall = num_same / len(ground_truth_tokens)
    return 2 * precision * recall / (precision + recall)


def evaluate_answer(prediction, gold_answers):
    if not gold_answers:
        return {"em": 0, "f1": 0.0}

    return {
        "em": max(exact_match_score(prediction, gold) for gold in gold_answers),
        "f1": max(f1_score(prediction, gold) for gold in gold_answers),
    }

In [77]:
single_metrics = evaluate_answer(
    qd_result["generated_answer"],
    test_record["answers"],
)

print("Exact Match:", single_metrics["em"])
print("F1:", round(single_metrics["f1"], 4))

Exact Match: 1
F1: 1.0


## Step 17 - Batch Evaluation Function

In [78]:
def evaluate_qd_records(records, limit=None):
    if limit is not None:
        records = records[:limit]

    results = []

    for index, record in enumerate(records, start=1):
        print(f"\nRunning {index} / {len(records)}")
        print("Question:", record["question"])

        try:
            rag_result = query_decomposition_rag(record["question"])
            prediction = rag_result["generated_answer"]
            metrics = evaluate_answer(prediction, record["answers"])

            retrieved = []
            for passage in rag_result["retrieved_passages"]:
                retrieved.append({
                    "passage_id": passage["passage_id"],
                    "title": passage["title"],
                    "reranker_score": passage["reranker_score"],
                    "retrieval_queries": passage.get("retrieval_queries", []),
                })

            results.append({
                "id": record["id"],
                "src": record["src"],
                "question": record["question"],
                "gold_answers": record["answers"],
                "subquestions": rag_result["subquestions"],
                "decomposition_fallback": rag_result["decomposition_fallback"],
                "candidate_count_before_rerank": rag_result["candidate_count_before_rerank"],
                "prediction": prediction,
                "em": metrics["em"],
                "f1": metrics["f1"],
                "retrieved_passages": retrieved,
                "error": None,
            })

            print("Gold:", record["answers"])
            print("Prediction:", prediction)
            print("EM:", metrics["em"])
            print("F1:", round(metrics["f1"], 4))

        except Exception as e:
            print("ERROR:", e)
            results.append({
                "id": record.get("id"),
                "src": record.get("src"),
                "question": record.get("question"),
                "gold_answers": record.get("answers"),
                "subquestions": [],
                "decomposition_fallback": None,
                "candidate_count_before_rerank": None,
                "prediction": None,
                "em": None,
                "f1": None,
                "retrieved_passages": [],
                "error": str(e),
            })

    return results

## Step 18 - QD9: Run 5 Questions First

If the Dev5 file already exists, it is loaded instead of making the API calls again.

In [79]:
if QD_DEV5_PATH.exists():
    qd_test5 = []
    with open(QD_DEV5_PATH, "r", encoding="utf-8") as f:
        for line in f:
            qd_test5.append(json.loads(line))
    print("Loaded saved QD Dev5 results:", len(qd_test5))
else:
    qd_test5 = evaluate_qd_records(beerqa_sample, limit=5)
    with open(QD_DEV5_PATH, "w", encoding="utf-8") as f:
        for result in qd_test5:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")
    print("Saved QD Dev5 results:", QD_DEV5_PATH)

Loaded saved QD Dev5 results: 5


In [80]:
successful_qd5 = [r for r in qd_test5 if r["error"] is None]

print("Questions completed:", len(successful_qd5))
print("Average EM:", round(sum(r["em"] for r in successful_qd5) / len(successful_qd5), 4))
print("Average F1:", round(sum(r["f1"] for r in successful_qd5) / len(successful_qd5), 4))

Questions completed: 5
Average EM: 0.4
Average F1: 0.5333


## Step 19 - QD10: Run All 20 Questions

The expensive Dev20 run is intentionally placed in its own cell. Run it only after Dev5 looks correct. It will be skipped automatically if a saved Dev20 result file already exists.

In [ ]:
if QD_DEV20_PATH.exists():
    qd_test20 = []
    with open(QD_DEV20_PATH, "r", encoding="utf-8") as f:
        for line in f:
            qd_test20.append(json.loads(line))
    print("Loaded saved QD Dev20 results:", len(qd_test20))
else:
    qd_test20 = []
    print("No saved QD Dev20 results yet.")

No saved QD Dev20 results yet.
Run the next cell only when you are ready.


In [82]:
# RUN THIS CELL ONLY WHEN YOU ARE READY FOR ALL 20 QUESTIONS.
# It may make about 40 DeepSeek calls: 20 decomposition + 20 final generation.

# qd_test20 = evaluate_qd_records(beerqa_sample, limit=20)
#
# with open(QD_DEV20_PATH, "w", encoding="utf-8") as f:
#     for result in qd_test20:
#         f.write(json.dumps(result, ensure_ascii=False) + "\n")
#
# print("Saved QD Dev20 results:", QD_DEV20_PATH)

In [83]:
if qd_test20:
    successful_qd20 = [r for r in qd_test20 if r["error"] is None]
    fallback_count = sum(bool(r.get("decomposition_fallback")) for r in successful_qd20)

    print("Total questions:", len(qd_test20))
    print("Successfully completed:", len(successful_qd20))
    print("Errors:", len(qd_test20) - len(successful_qd20))
    print("Average EM:", round(sum(r["em"] for r in successful_qd20) / len(successful_qd20), 4))
    print("Average F1:", round(sum(r["f1"] for r in successful_qd20) / len(successful_qd20), 4))
    print("Decomposition fallbacks:", fallback_count)

## Step 20 - QD11: Development Retrieval Metrics

For consistency with the Standard RAG Dev20 notebook, calculate **Title Recall@10** and **Title Precision@10**. These are development title/document metrics, not final passage-level BeerQA metrics.

In [84]:
def get_gold_titles(record):
    titles = []
    for item in record.get("context", []):
        if isinstance(item, list) and len(item) >= 1:
            title = normalize_title(item[0])
            if title and title not in titles:
                titles.append(title)
    return titles


def get_retrieved_titles(result, top_k=10):
    return [
        normalize_title(p["title"])
        for p in result["retrieved_passages"][:top_k]
    ]


def title_recall_at_k(retrieved_titles, gold_titles):
    gold_set = set(gold_titles)
    if not gold_set:
        return 0.0
    return len(set(retrieved_titles) & gold_set) / len(gold_set)


def title_precision_at_k(retrieved_titles, gold_titles):
    unique_retrieved = list(dict.fromkeys(retrieved_titles))
    if not unique_retrieved:
        return 0.0
    return len(set(unique_retrieved) & set(gold_titles)) / len(unique_retrieved)

In [85]:
if qd_test20:
    retrieval_eval = []

    for record, result in zip(beerqa_sample, qd_test20):
        gold_titles = get_gold_titles(record)
        retrieved_titles = get_retrieved_titles(result, top_k=10)
        recall10 = title_recall_at_k(retrieved_titles, gold_titles)
        precision10 = title_precision_at_k(retrieved_titles, gold_titles)

        result["gold_titles"] = gold_titles
        result["title_recall@10"] = recall10
        result["title_precision@10"] = precision10

        retrieval_eval.append({
            "title_recall@10": recall10,
            "title_precision@10": precision10,
        })

    print("Questions evaluated:", len(retrieval_eval))
    print("Average Title Recall@10:", round(sum(x["title_recall@10"] for x in retrieval_eval) / len(retrieval_eval), 4))
    print("Average Title Precision@10:", round(sum(x["title_precision@10"] for x in retrieval_eval) / len(retrieval_eval), 4))

## Step 21 - Save Fully Evaluated Dev20 Results

In [86]:
if qd_test20:
    with open(QD_DEV20_EVAL_PATH, "w", encoding="utf-8") as f:
        for result in qd_test20:
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    print("Saved evaluated QD results:", QD_DEV20_EVAL_PATH)

## Step 22 - Compare with the Saved Standard RAG Baseline

This reads the two evaluated JSONL files and prints the same four development metrics side by side.

In [87]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def summarize(rows):
    rows = [r for r in rows if r.get("error") is None]
    summary = {
        "EM": sum(r["em"] for r in rows) / len(rows),
        "F1": sum(r["f1"] for r in rows) / len(rows),
    }

    if all("title_recall@10" in r for r in rows):
        summary["Title Recall@10"] = sum(r["title_recall@10"] for r in rows) / len(rows)
    if all("title_precision@10" in r for r in rows):
        summary["Title Precision@10"] = sum(r["title_precision@10"] for r in rows) / len(rows)

    return summary


if STANDARD_EVAL_PATH.exists() and QD_DEV20_EVAL_PATH.exists():
    standard_summary = summarize(load_jsonl(STANDARD_EVAL_PATH))
    qd_summary = summarize(load_jsonl(QD_DEV20_EVAL_PATH))

    print(f"{'Metric':<22}{'Standard RAG':>18}{'Query Decomp RAG':>20}")
    print("-" * 60)

    for metric in ["EM", "F1", "Title Recall@10", "Title Precision@10"]:
        print(
            f"{metric:<22}"
            f"{standard_summary.get(metric, float('nan')):>18.4f}"
            f"{qd_summary.get(metric, float('nan')):>20.4f}"
        )
else:
    print("Comparison will be available after both evaluated Dev20 files exist.")

Comparison will be available after both evaluated Dev20 files exist.
